In [ ]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG — CHANGE THIS ONLY
# =========================
# CSV_PATH = "/content/drive/MyDrive/Human Evals/Qwen_exp17_human_eval.csv"
CSV_PATH = "/content/drive/MyDrive/Human Evals/gpt4.1_human_eval.csv"

In [ ]:
# ===============================
# 1️ Install dependencies
# ===============================
!pip install datasets pandas

# ===============================
# 2️ Mount Google Drive
# ===============================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG & SETUP
# =========================
csv_file = Path(CSV_PATH)
if not csv_file.exists():
    # checking for common fallback
    if Path(f"data/{CSV_PATH}").exists():
        csv_file = Path(f"data/{CSV_PATH}")
    else:
        raise FileNotFoundError(f"CSV file not found at: {CSV_PATH}")

df = pd.read_csv(csv_file)

# =========================
# HELPERS
# =========================
def safe_div(n, d, default="NA"):
    return round(n / d, 4) if d and d > 0 else default

def error_rate_percent(series):
    """Calculates the percentage of 1s in a binary series (ignoring NAs)."""
    s = series.dropna()
    if len(s) == 0:
        return "NA"
    return round((s.sum() / len(s)) * 100, 2)

def invert_error_rate(series):
    """For positive metrics (e.g. format_compliance), converts to error rate (1 - val)."""
    s = series.dropna()
    if len(s) == 0:
        return "NA"
    return round(((len(s) - s.sum()) / len(s)) * 100, 2)

# =========================
# 1. NO-RULE DETECTION ANALYSIS
#    (Confusion Matrix Logic)
# =========================
no_rule_stats = {}

if "gold_has_business_rules" in df.columns and "rules_present" in df.columns:
    negatives = df[df["gold_has_business_rules"] == 0]
    total_negatives = len(negatives)

    false_positives = negatives[negatives["rules_present"] == 1].shape[0]

    positives = df[df["gold_has_business_rules"] == 1]
    total_positives = len(positives)

    false_negatives = positives[positives["rules_present"] == 0].shape[0]

    no_rule_stats = {
        "Gold 'No Rules' Count": total_negatives,
        "False Positive (Hallucinated Presence)": f"{false_positives} ({safe_div(false_positives, total_negatives)*100}%)",
        "Gold 'Has Rules' Count": total_positives,
        "False Negative (Missed Presence)": f"{false_negatives} ({safe_div(false_negatives, total_positives)*100}%)",
    }
else:
    no_rule_stats = {"Error": "Missing columns 'gold_has_business_rules' or 'rules_present'"}

# =========================
# 2. AGGREGATED RULE COUNTS
# =========================
count_metrics = [
    "gold_rule_count",
    "extracted_rule_count",
    "correct_rule_count",
    "missed_rule_count",
    "hallucinated_rule_count",
    "duplicate_rule_count",
    "paraphrase_duplicate_count",
]

rule_sums = {
    col: int(df[col].dropna().sum())
    for col in count_metrics
    if col in df.columns
}

total_gold = rule_sums.get("gold_rule_count", 0)
total_extracted = rule_sums.get("extracted_rule_count", 0)

# =========================
# 3. CATEGORICAL ANALYSIS
#    (Language Level & Formats)
# =========================
cat_stats = {}

# Wording Level (Technical vs Business)
if "wording_level" in df.columns:
    counts = df["wording_level"].value_counts(normalize=True) * 100
    cat_stats["Wording Level"] = counts.to_dict() 

# Bullet Format
if "bullet_format" in df.columns:
    counts = df["bullet_format"].value_counts(normalize=True) * 100
    cat_stats["Bullet Format"] = counts.to_dict()

# =========================
# 4. ERROR CLASS REPORT
# =========================

error_report = {
    "1. Extraction Errors": {
        "Missed Rule Rate (Count-based)": f"{safe_div(rule_sums.get('missed_rule_count', 0), total_gold) * 100}%",
        "Hallucination Rate (Count-based)": f"{safe_div(rule_sums.get('hallucinated_rule_count', 0), total_extracted) * 100}%",
        "Rule Deferral Rate (Row %)": error_rate_percent(df.get("rule_deferral")),
        "Conditions Missed (Row %)": error_rate_percent(df.get("condition_missed")),
        "Actions Missed (Row %)": error_rate_percent(df.get("action_missed")),
    },

    "2. Redundancy Errors": {
        "Duplicate Density (per extracted rule)": safe_div(rule_sums.get("duplicate_rule_count", 0), total_extracted),
        "Paraphrase Loops (Total Count)": rule_sums.get("paraphrase_duplicate_count", 0),
        "Last Line Looping (Row %)": error_rate_percent(df.get("last_line_looping")),
    },

    "3. Formatting Errors": {
        "Format Non-Compliance (Row %)": invert_error_rate(df.get("format_compliance")),
        "Placeholder Titles Only (Row %)": error_rate_percent(df.get("placeholder_titles_only")),
        "Repeated Headings (Row %)": error_rate_percent(df.get("repeated_headings")),
        "Unnecessary End Markers (Row %)": error_rate_percent(df.get("unnecessary_end_markers")),
        "Markdown Structure Quality (Avg Score 0-2)": round(df["markdown_structure_quality"].mean(), 2) if "markdown_structure_quality" in df else "NA",
    },

    "4. Language Errors": {
        "Implementation Leakage (Row %)": error_rate_percent(df.get("implementation_leakage")),
        "Non-Entity Centric Rules (Row %)": invert_error_rate(df.get("entity_centric_rules")),
        "Technical Artifact Rules (Row %)": error_rate_percent(df.get("technical_artifact_rules")),
        "Technical Wording Prevalence": f"{round(cat_stats.get('Wording Level', {}).get('technical', 0), 2)}%"
    },

    "5. Degeneration Errors": {
        "Endless Repetition (Row %)": error_rate_percent(df.get("endless_repetition")),
        "Instruction Echoing (Row %)": error_rate_percent(df.get("instruction_echo")),
        "Explains 'What is a rule' (Row %)": error_rate_percent(df.get("explains_what_is_rule")),
        "Empty After Title (Row %)": error_rate_percent(df.get("empty_after_title")),
    }
}

# =========================
# 5. PRINT REPORT
# =========================

print(f"\n{'='*40}")
print(f"{'EVALUATION REPORT':^40}")
print(f"{'='*40}")

print(f"\n--- A. No-Rule Detection (Confusion Matrix) ---")
for k, v in no_rule_stats.items():
    print(f"{k:40s}: {v}")

print(f"\n--- B. Global Rule Counts ---")
print(f"{'Total Gold Rules':40s}: {total_gold}")
print(f"{'Total Extracted Rules':40s}: {total_extracted}")
print(f"{'Total Correct Rules':40s}: {rule_sums.get('correct_rule_count', 0)}")

print(f"\n--- C. Detailed Error Class Analysis ---")
for category, metrics in error_report.items():
    print(f"\n[{category}]")
    for name, value in metrics.items():
        print(f"  • {name:35s}: {value}")

print(f"\n--- D. Categorical Distribution ---")
for cat, stats in cat_stats.items():
    print(f"\n{cat}:")
    for k, v in stats.items():
        print(f"  • {k}: {round(v, 2)}%")

print(f"\n{'='*40}")
print("END OF REPORT")
print(f"{'='*40}")


           EVALUATION REPORT            

--- A. No-Rule Detection (Confusion Matrix) ---
Error                                   : Missing columns 'gold_has_business_rules' or 'rules_present'

--- B. Global Rule Counts ---
Total Gold Rules                        : 148
Total Extracted Rules                   : 201
Total Correct Rules                     : 75

--- C. Detailed Error Class Analysis ---

[1. Extraction Errors]
  • Missed Rule Rate (Count-based)     : 60.809999999999995%
  • Hallucination Rate (Count-based)   : 7.46%
  • Rule Deferral Rate (Row %)         : 0.0
  • Conditions Missed (Row %)          : 30.0
  • Actions Missed (Row %)             : 25.0

[2. Redundancy Errors]
  • Duplicate Density (per extracted rule): 0.0
  • Paraphrase Loops (Total Count)     : 9
  • Last Line Looping (Row %)          : 0.0

[3. Formatting Errors]
  • Format Non-Compliance (Row %)      : 0.0
  • Placeholder Titles Only (Row %)    : 0.0
  • Repeated Headings (Row %)          : 0.0
  • Unne

In [ ]:
# =========================
# SAFETY CHECK
# =========================
csv_file = Path(CSV_PATH)
if not csv_file.exists():
    raise FileNotFoundError(f"CSV file not found at: {CSV_PATH}")

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(csv_file)

# =========================
# HELPERS
# =========================
def mean_ignore_na(series):
    return series.dropna().mean()

def sum_ignore_na(series):
    return series.dropna().sum()

def error_rate(series):
    s = series.dropna()
    if len(s) == 0:
        return None
    return round((s.sum() / len(s)) * 100, 2)

def safe(v):
    return "NA" if v is None or pd.isna(v) else v

# =========================
# METRIC COLUMNS
# =========================
metric_columns = [
    # Presence & Termination
    "output_non_empty", "premature_termination", "endless_repetition",
    "last_line_looping", "repeated_headings", "unnecessary_end_markers",

    # Formatting
    "has_title", "title_correct", "format_compliance",
    "instruction_echo", "code_echo",
    "rule_grouping_present", "markdown_structure_quality",

    # Semantic Quality
    "over_generalized_rules", "under_specified_rules",
    "condition_missed", "action_missed",
    "rule_deferral", "over_summarization",

    # Language
    "implementation_leakage", "entity_centric_rules",

    # Degeneration
    "explains_what_is_rule", "placeholder_titles_only", "empty_after_title",

    # Repair
    "repair_effort_level", "salvageable_output",
]

# =========================
# 1. METRIC SUMMARY
#    (MEAN + SUM)
# =========================
metric_summary = {}

for col in metric_columns:
    if col in df.columns:
        metric_summary[col] = {
            "mean": mean_ignore_na(df[col]),
            "sum": sum_ignore_na(df[col])
        }

# =========================
# 2. AGGREGATED RULE COUNTS
# =========================
count_metrics = [
    "gold_rule_count",
    "extracted_rule_count",
    "correct_rule_count",
    "missed_rule_count",
    "hallucinated_rule_count",
    "duplicate_rule_count",
    "paraphrase_duplicate_count",
]

rule_counts = {
    col: int(df[col].dropna().sum())
    for col in count_metrics
    if col in df.columns
}

# =========================
# CORRECT ERROR METRICS
# =========================

total_gold = rule_counts.get("gold_rule_count", 0)
total_extracted = rule_counts.get("extracted_rule_count", 0)

error_report = {
    "Extraction Errors": {
        "Missed rule rate (%)": (
            round(100 * rule_counts["missed_rule_count"] / total_gold, 2)
            if total_gold > 0 else "NA"
        ),
        "Hallucination rate (%)": (
            round(100 * rule_counts["hallucinated_rule_count"] / total_extracted, 2)
            if total_extracted > 0 else "NA"
        ),
        "Rule deferral (sample %)": error_rate(df.get("rule_deferral")),
    },

    "Redundancy Errors": {
        "Duplicate density (per extracted rule)": (
            round(rule_counts["duplicate_rule_count"] / total_extracted, 2)
            if total_extracted > 0 else "NA"
        ),
        "Paraphrase duplicates (total)": rule_counts["paraphrase_duplicate_count"],
    },

    "Formatting Errors": {
        "Format non-compliance (sample %)": (
            error_rate(1 - df["format_compliance"])
            if "format_compliance" in df else "NA"
        ),
        "Repeated headings (sample %)": error_rate(df.get("repeated_headings")),
        "Placeholder-only output (sample %)": error_rate(df.get("placeholder_titles_only")),
    },

    "Language Errors": {
        "Implementation leakage (sample %)": error_rate(df.get("implementation_leakage")),
        "Non–entity-centric rules (sample %)": (
            error_rate(1 - df["entity_centric_rules"])
            if "entity_centric_rules" in df else "NA"
        ),
    },

    "Degeneration Errors": {
        "Endless repetition (sample %)": error_rate(df.get("endless_repetition")),
        "Instruction echoing (sample %)": error_rate(df.get("instruction_echo")),
        "Explains instead of extracting (sample %)": error_rate(df.get("explains_what_is_rule")),
    },
}


# =========================
# 4. PRINT REPORT
# =========================
print("\n==============================")
print("FINAL METRIC SUMMARY")
print("(MEAN + SUM, NA-SAFE)")
print("==============================")
for k, v in metric_summary.items():
    print(f"{k:35s} | mean: {safe(round(v['mean'], 3))} | sum: {safe(v['sum'])}")

print("\n==============================")
print("AGGREGATED RULE COUNTS")
print("==============================")
for k, v in rule_counts.items():
    print(f"{k:35s}: {v}")

print("\n==============================")
print("MODEL PERFORMANCE REPORT")
print("(Error-Class Based)")
print("==============================")
for category, metrics in error_report.items():
    print(f"\n{category}")
    for name, value in metrics.items():
        print(f"  - {name:30s}: {safe(value)}")

print("\n==============================")
print("END OF REPORT")
print("==============================")



FINAL METRIC SUMMARY
(MEAN + SUM, NA-SAFE)
output_non_empty                    | mean: 1.0 | sum: 20
premature_termination               | mean: 0.1 | sum: 2
endless_repetition                  | mean: 0.1 | sum: 2
last_line_looping                   | mean: 0.1 | sum: 2
repeated_headings                   | mean: 0.0 | sum: 0
unnecessary_end_markers             | mean: 0.0 | sum: 0
has_title                           | mean: 0.75 | sum: 15
title_correct                       | mean: 1.0 | sum: 15.0
format_compliance                   | mean: 0.95 | sum: 19
instruction_echo                    | mean: 0.0 | sum: 0
code_echo                           | mean: 0.0 | sum: 0
rule_grouping_present               | mean: 0.4 | sum: 8
markdown_structure_quality          | mean: 1.35 | sum: 27
over_generalized_rules              | mean: 0.105 | sum: 2.0
under_specified_rules               | mean: 0.15 | sum: 3
condition_missed                    | mean: 0.1 | sum: 2
action_missed                